# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [ ]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


def get_completion_from_messages(message, model="gpt-3.5-turbo", temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

In [3]:
messages =  [
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},
{'role':'user', 'content':'tell me a joke'},
{'role':'assistant', 'content':'Why did the chicken cross the road'},
{'role':'user', 'content':'I don\'t know'}  ]

In [7]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side, good sir! 'Tis a classic jest that oft amuseth the masses.


In [10]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's nice to meet you. How are you doing today?


In [11]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have access to that information. You may need to remind me of your name.


In [14]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant.

In [15]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))

    return pn.Column(*panels)


In [16]:
import panel as pn  # library for building interactive dashboards and GUIs
pn.extension() # loads Panel’s JavaScript and CSS so widgets like text boxes and buttons work in your notebook or web app.

panels = [] # This list will store the chat messages that are displayed in the dashboard.

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…') # Creates a text input box for the user to type their message.
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation) # pn.bind connects the button click to the function collect_messages

# Building the Dashboard Layout
dashboard = pn.Column(
    inp, # text input at top
    pn.Row(button_conversation), # button displayed below input
    pn.panel(interactive_conversation, # the chat messages displayed in a scrollable panel
             loading_indicator=True, # shows a spinner while waiting for AI
             height=300), # panel height
)

dashboard # When you run this cell, Panel renders the interactive UI in your notebook.

/tmp/ipython-input-198781390.py:2: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [17]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},

response = get_completion_from_messages(messages, temperature=0)
print(response)

```json
{
    "pizza": {
        "type": "pepperoni pizza",
        "size": "large"
    },
    "toppings": [
        "extra cheese",
        "mushrooms"
    ],
    "drinks": [
        {
            "type": "coke",
            "size": "medium"
        }
    ],
    "sides": [
        {
            "type": "fries",
            "size": "regular"
        }
    ],
    "total price": 23.45
}
```


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [18]:
messages.append({
    'role': 'system',
    'content': 'Summarize the user pizza order in JSON. Include pizza type, size, toppings, drinks, sides, and total price.'
})
response = get_completion_from_messages(messages, temperature=0)
print(response)

{
    "pizza": {
        "type": "pepperoni pizza",
        "size": "large"
    },
    "toppings": [
        "extra cheese",
        "mushrooms"
    ],
    "drinks": [
        {
            "type": "coke",
            "size": "medium"
        }
    ],
    "sides": [],
    "total price": 18.45
}


In [23]:
messages.append({
    'role': 'system',
    'content': 'Summarize the user pizza order in JSON. act as if it was a delivery and include it in the json ,and act accordingly'
})
response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large"
  },
  "toppings": ["extra cheese", "mushrooms"],
  "drinks": [
    {
      "type": "coke",
      "size": "medium"
    }
  ],
  "sides": [],
  "total price": 18.45,
  "delivery": true
}


In [24]:
messages.append({
    'role': 'system',
    'content': '''
Generate a JSON summary of the pizza order. Be precise: do not invent items or prices. Make sure all fields exist and prices are accurate.
Return only JSON, no extra text.
'''
})
response = get_completion_from_messages(messages, temperature=0)
print(response)


{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large"
  },
  "toppings": ["extra cheese", "sausage"],
  "drinks": [
    {
      "type": "coke",
      "size": "medium"
    }
  ],
  "sides": [],
  "total price": 18.95,
  "delivery": true
}


### Analyze variations:

- Version 1: simple, worked, but no delivery field

- Version 2: delivery included, but otherwise correct

- Version 3: AI changed toppings (sausage instead of mushrooms) → this is a hallucination / inconsistency

### Key Learnings:

Being explicit reduces hallucinations but isn’t perfect

Adding constraints (do not invent items or prices) is helpful

Always check AI output; even with deterministic temperature=0, errors can happen